# NYC 311 snapshot — profile

**Question:** what is actually in a month of `HEAT/HOT WATER` requests, and can we build a
slow-resolution risk label from it?

**Source:** Socrata dataset `erm2-nwe9` via `scripts`-free direct HTTP — no API key needed.
**Snapshot:** 5,000 rows, `sha256` recorded below.

In [1]:
import hashlib, json, time
import httpx
import pandas as pd
from resolveiq.config import REPO_ROOT, get_settings
import numpy as np
import datetime as dt

In [2]:
settings = get_settings()
url = f"{settings.nyc311_base_url}/{settings.nyc311_dataset_id}.json"
params = {
    "$select": ("unique_key, created_date, closed_date, complaint_type, descriptor, "
                "location_type, street_name, city, borough, status, latitude, longitude"),
    "$where": ("lower(complaint_type) = 'heat/hot water' "
               "and created_date >= '2024-01-01T00:00:00' "
               "and created_date < '2024-02-01T00:00:00'"),
    "$order": "created_date, unique_key",
    "$limit": "5000",
}
rows = httpx.get(url, params=params, timeout=60.0).raise_for_status().json()
len(rows)

5000

In [3]:
raw_dir = REPO_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
out_file = raw_dir / "heat_2024_01_5000.json"
payload = json.dumps(rows, indent=2)
out_file.write_text(payload, encoding="utf-8")
{
    "url": url, "params": params, "rows": len(rows),
    "bytes": out_file.stat().st_size,
    "sha256": hashlib.sha256(payload.encode("utf-8")).hexdigest(),
    "fetched_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}

{'url': 'https://data.cityofnewyork.us/resource/erm2-nwe9.json',
 'params': {'$select': 'unique_key, created_date, closed_date, complaint_type, descriptor, location_type, street_name, city, borough, status, latitude, longitude',
  '$where': "lower(complaint_type) = 'heat/hot water' and created_date >= '2024-01-01T00:00:00' and created_date < '2024-02-01T00:00:00'",
  '$order': 'created_date, unique_key',
  '$limit': '5000'},
 'rows': 5000,
 'bytes': 2264248,
 'sha256': '3e2cf9aa6ba720b122285b15c663edf73629431b74c98708e51060450df5c5b3',
 'fetched_at_utc': '2026-09-23T03:04:41Z'}

In [4]:
df = pd.read_json(out_file)
df.shape

(5000, 12)

## 1. Schema and completeness

In [5]:
df.dtypes

unique_key          int64
created_date          str
closed_date           str
complaint_type        str
descriptor            str
location_type         str
street_name           str
city                  str
borough               str
status                str
latitude          float64
longitude         float64
dtype: object

In [6]:
df.isna().sum()

unique_key        0
created_date      0
closed_date       1
complaint_type    0
descriptor        0
location_type     0
street_name       0
city              0
borough           0
status            0
latitude          0
longitude         0
dtype: int64

In [7]:
df["status"].value_counts()

status
Closed    4999
Open         1
Name: count, dtype: int64

In [8]:
df[["created_date", "closed_date"]].agg(["min", "max"])

,created_date,closed_date
min,2024-01-01T00:01:45.000,2024-01-01T09:58:04.000
max,2024-01-04T21:51:29.000,2025-07-07T16:08:14.000


In [9]:
df["descriptor"].value_counts().head(12)

descriptor
ENTIRE BUILDING    3186
APARTMENT ONLY     1814
Name: count, dtype: int64

In [10]:
df["location_type"].value_counts()

location_type
RESIDENTIAL BUILDING    5000
Name: count, dtype: int64

## 2. Time coverage and resolution duration

In [11]:
df["created_date"] = pd.to_datetime(df["created_date"])
df["closed_date"] = pd.to_datetime(df["closed_date"])
df["resolution_hours"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
print(df["resolution_hours"].describe())

count     4999.000000
mean        32.099942
std        188.019870
min          0.000000
25%         15.661806
50%         26.015556
75%         41.485972
max      13253.573056
Name: resolution_hours, dtype: float64


In [12]:
first, last = df["created_date"].min(), df["created_date"].max()
print(f"created : {first}  ->  {last}")
print(f"span    : {last - first}")
print(f"days    : {df['created_date'].dt.date.nunique()} distinct dates out of 31 in January")

created : 2024-01-01 00:01:45  ->  2024-01-04 21:51:29
span    : 3 days 21:49:44
days    : 4 distinct dates out of 31 in January


In [13]:
names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
counts = df["created_date"].dt.dayofweek.value_counts().sort_index()
print({names[i]: int(c) for i, c in counts.items()})

{'Mon': 850, 'Tue': 1496, 'Wed': 1327, 'Thu': 1327}


In [14]:
print(df["descriptor"].value_counts().head(12).to_dict())
print(df["location_type"].value_counts().to_dict())
print(df["borough"].value_counts().to_dict())

{'ENTIRE BUILDING': 3186, 'APARTMENT ONLY': 1814}
{'RESIDENTIAL BUILDING': 5000}
{'BRONX': 1895, 'MANHATTAN': 1196, 'BROOKLYN': 1192, 'QUEENS': 643, 'STATEN ISLAND': 74}


In [15]:
print(f"under 1 hour : {int((df['resolution_hours'] < 1).sum())}  "
      f"({(df['resolution_hours'] < 1).mean():.1%})")
print(f"exactly 0 min: {int((df['resolution_hours'] == 0).sum())}")

under 1 hour : 49  (1.0%)
exactly 0 min: 1


In [16]:
long = df[df["resolution_hours"] > 24 * 30]
print(f"over 30 days : {len(long)}  ({long['resolution_hours'].mean():.0f} h mean)")
print(long[["unique_key", "created_date", "closed_date", "resolution_hours"]].head(3).to_string())

over 30 days : 1  (13254 h mean)
      unique_key        created_date         closed_date  resolution_hours
1348    59903995 2024-01-02 10:33:51 2025-07-07 16:08:14      13253.573056


In [17]:
for q in [0.5, 0.75, 0.9, 0.95, 0.99]:
    print(f"p{int(q * 100):<3}: {df['resolution_hours'].quantile(q):8.1f} h")

p50 :     26.0 h
p75 :     41.5 h
p90 :     54.1 h
p95 :     61.3 h
p99 :     84.7 h


## 4. Re-pull: a sample spread across the whole month

The first pull is biased by construction (oldest-first, capped at 5,000 = 4 days). This pulls
`per_day` rows for each of January's 31 days, and adds `descriptor_2`, which carries the symptom.

In [18]:
days = [dt.date(2024, 1, 1) + dt.timedelta(days=n) for n in range(31)]
print(len(days), days[0], "->", days[-1])

31 2024-01-01 -> 2024-01-31


In [19]:
per_day = 160
rows = []
started = time.perf_counter()
for day in days:
    params = {
        "$select": ("unique_key, created_date, closed_date, complaint_type, descriptor, descriptor_2, "
                    "location_type, street_name, city, borough, status, latitude, longitude"),
        "$where": (f"lower(complaint_type) = 'heat/hot water' "
                   f"and created_date >= '{day.isoformat()}T00:00:00' "
                   f"and created_date < '{(day + dt.timedelta(days=1)).isoformat()}T00:00:00'"),
        "$order": "created_date, unique_key",
        "$limit": str(per_day),
    }
    response = httpx.get(url, params=params, timeout=60.0)
    response.raise_for_status()
    rows.extend(response.json())
print(f"{len(rows)} rows in {time.perf_counter() - started:.1f}s  ({len(days)} requests)")

4960 rows in 46.6s  (31 requests)


In [20]:
per_day = 160
rows = []
started = time.perf_counter()
for day in days:
    params = {
        "$select": ("unique_key, created_date, closed_date, complaint_type, descriptor, descriptor_2, "
                    "location_type, street_name, city, borough, status, latitude, longitude"),
        "$where": (f"lower(complaint_type) = 'heat/hot water' "
                   f"and created_date >= '{day.isoformat()}T00:00:00' "
                   f"and created_date < '{(day + dt.timedelta(days=1)).isoformat()}T00:00:00'"),
        "$order": "created_date, unique_key",
        "$limit": str(per_day),
    }
    response = httpx.get(url, params=params, timeout=60.0)
    response.raise_for_status()
    rows.extend(response.json())
print(f"{len(rows)} rows in {time.perf_counter() - started:.1f}s  ({len(days)} requests)")

4960 rows in 29.6s  (31 requests)


In [21]:
per_day = 160
rows = []
started = time.perf_counter()
for day in days:
    params = {
        "$select": ("unique_key, created_date, closed_date, complaint_type, descriptor, descriptor_2, "
                    "location_type, street_name, city, borough, status, latitude, longitude"),
        "$where": (f"lower(complaint_type) = 'heat/hot water' "
                   f"and created_date >= '{day.isoformat()}T00:00:00' "
                   f"and created_date < '{(day + dt.timedelta(days=1)).isoformat()}T00:00:00'"),
        "$order": "created_date, unique_key",
        "$limit": str(per_day),
    }
    response = httpx.get(url, params=params, timeout=60.0)
    response.raise_for_status()
    rows.extend(response.json())
print(f"{len(rows)} rows in {time.perf_counter() - started:.1f}s  ({len(days)} requests)")

4960 rows in 31.4s  (31 requests)


In [22]:
snap = pd.DataFrame(rows)
snap["created_date"] = pd.to_datetime(snap["created_date"])
counts = snap.groupby(snap["created_date"].dt.date).size()
print(f"days covered : {len(counts)}")
print(f"rows per day : min {counts.min()}  median {int(counts.median())}  max {counts.max()}")
print(f"descriptor_2 blank in : {int(snap['descriptor_2'].isna().sum())} of {len(snap)}")
print(snap["descriptor_2"].value_counts().head(10).to_dict())

days covered : 31
rows per day : min 160  median 160  max 160
descriptor_2 blank in : 0 of 4960
{'NO HEAT': 3318, 'NO HEAT AND NO HOT WATER': 1342, 'NO HOT WATER': 300}


In [23]:
out_file = REPO_ROOT / "data" / "raw" / "heat_2024_01_stratified.json"
payload = json.dumps(rows, indent=2)
out_file.write_text(payload, encoding="utf-8")
print(f"sha256 {hashlib.sha256(payload.encode('utf-8')).hexdigest()[:16]}...  {out_file.stat().st_size:,} bytes")

sha256 17ebb68746c4a2fd...  2,428,687 bytes


## Daily counts: API-compatible approach

Socrata rejected the attempted `date_trunc` query, so this notebook does not depend on
database-side date functions. The next cell queries one calendar day at a time, which is
slower but explicit, reproducible, and gives us a complete 31-day January profile.

In [24]:
# Query one calendar day at a time because Socrata does not support date_trunc here.
volume_rows = []
started = time.perf_counter()
for day in days:
    params = {
        "$select": "count(*) as n",
        "$where": (f"lower(complaint_type) = 'heat/hot water' "
                   f"and created_date >= '{day.isoformat()}T00:00:00' "
                   f"and created_date < '{(day + dt.timedelta(days=1)).isoformat()}T00:00:00'"),
    }
    response = httpx.get(url, params=params, timeout=60.0)
    response.raise_for_status()
    volume_rows.append({"day": day.isoformat(), "n": int(response.json()[0]["n"])})

volume = pd.DataFrame(volume_rows)
volume["day"] = pd.to_datetime(volume["day"])
print(f"{len(volume)} days  total {volume['n'].sum():,}  in {time.perf_counter() - started:.1f}s")
print("quietest 3:")
print(volume.nsmallest(3, "n").to_string(index=False))
print("busiest 3:")
print(volume.nlargest(3, "n").to_string(index=False))

31 days  total 51,325  in 38.2s
quietest 3:
       day   n
2024-01-27 798
2024-01-01 850
2024-01-25 855
busiest 3:
       day    n
2024-01-18 3784
2024-01-17 3298
2024-01-21 2993


In [25]:
volume["weekday"] = volume["day"].dt.day_name().str[:3]
print(volume.groupby("weekday")["n"].agg(["count", "mean", "min", "max"]).round(0).to_string())
print(f"\n3 busiest days = {volume.nlargest(3, 'n')['n'].sum() / volume['n'].sum():.1%} of the month")

         count    mean   min   max
weekday                           
Fri          4  1496.0   860  2244
Mon          5  1668.0   850  2704
Sat          4  1571.0   798  2901
Sun          4  1753.0  1026  2993
Thu          4  1806.0   855  3784
Tue          5  1650.0  1251  2669
Wed          5  1646.0  1118  3298

3 busiest days = 19.6% of the month


In [26]:
top = volume.nlargest(8, "n")
print(top[["day", "weekday", "n"]].to_string(index=False))
print("\nJan days in that top 8:", sorted(top["day"].dt.day.tolist()))

       day weekday    n
2024-01-18     Thu 3784
2024-01-17     Wed 3298
2024-01-21     Sun 2993
2024-01-20     Sat 2901
2024-01-15     Mon 2704
2024-01-16     Tue 2669
2024-01-19     Fri 2244
2024-01-22     Mon 2218

Jan days in that top 8: [15, 16, 17, 18, 19, 20, 21, 22]


In [27]:
frac = 0.10
targets = {d.date(): max(40, int(round(n * frac))) for d, n in zip(volume["day"], volume["n"])}
print(f"planned {sum(targets.values()):,} rows across {len(targets)} days, 10% of every day")

prop_rows = []
started = time.perf_counter()
for day, want in targets.items():
    params = {
        "$select": ("unique_key, created_date, closed_date, complaint_type, descriptor, descriptor_2, "
                    "location_type, street_name, city, borough, status, latitude, longitude"),
        "$where": (f"lower(complaint_type) = 'heat/hot water' "
                   f"and created_date >= '{day.isoformat()}T00:00:00' "
                   f"and created_date < '{(day + dt.timedelta(days=1)).isoformat()}T00:00:00'"),
        "$order": "created_date, unique_key",
        "$limit": str(want),
    }
    response = httpx.get(url, params=params, timeout=60.0)
    response.raise_for_status()
    prop_rows.extend(response.json())
print(f"{len(prop_rows):,} rows in {time.perf_counter() - started:.1f}s")

planned 5,135 rows across 31 days, 10% of every day
5,135 rows in 29.2s


In [28]:
prop = pd.DataFrame(prop_rows)
prop["created_date"] = pd.to_datetime(prop["created_date"])
got = prop.groupby(prop["created_date"].dt.date).size()
planned = pd.Series(targets).sort_index()
print(f"days {len(got)}   rows/day min {got.min()} max {got.max()}")
print(f"actual / planned   min {(got / planned).min():.2f}   max {(got / planned).max():.2f}")

out_file = REPO_ROOT / "data" / "raw" / "heat_2024_01_prop10.json"
payload = json.dumps(prop_rows, indent=2)
out_file.write_text(payload, encoding="utf-8")
print(f"sha256 {hashlib.sha256(payload.encode('utf-8')).hexdigest()[:16]}...  {out_file.stat().st_size:,} bytes")

days 31   rows/day min 80 max 378
actual / planned   min 1.00   max 1.00
sha256 1a019a8090da0fcb...  2,512,697 bytes


In [29]:
def count(where):
    response = httpx.get(url, params={"$select": "count(*) as n", "$where": where}, timeout=60.0)
    response.raise_for_status()
    return int(response.json()[0]["n"])

event = "created_date >= '2024-01-15T00:00:00' and created_date < '2024-01-23T00:00:00'"
quiet = "created_date >= '2024-01-24T00:00:00' and created_date < '2024-01-31T00:00:00'"
heat = "lower(complaint_type) = 'heat/hot water' and "

all_event, all_quiet = count(event), count(quiet)
heat_event, heat_quiet = count(heat + event), count(heat + quiet)
print(f"all 311   {all_event/8:>8,.0f}/day   vs {all_quiet/7:>8,.0f}/day   ratio {all_event/8/(all_quiet/7):.2f}")
print(f"heat only {heat_event/8:>8,.0f}/day   vs {heat_quiet/7:>8,.0f}/day   ratio {heat_event/8/(heat_quiet/7):.2f}")

all 311      9,974/day   vs    9,017/day   ratio 1.11
heat only    2,851/day   vs    1,045/day   ratio 2.73


In [30]:
meta_url = url.replace("/resource/", "/api/views/").replace(".json", "")
meta = httpx.get(meta_url, timeout=60.0)
meta.raise_for_status()
for column in meta.json()["columns"]:
    if column["fieldName"] in {"descriptor", "descriptor_2", "created_date", "closed_date", "status"}:
        note = column.get("description") or "(source gives no description)"
        print(f"{column['fieldName']:<14} {column.get('typeName', ''):<22} :: {note[:150]}")

created_date                          :: Date SR  was created

closed_date                           :: Date SR was closed by responding agency

descriptor                            :: This is associated to the Problem (formerly Complaint Type), and provides further detail on the incident or condition. Problem Detail (formerly Descri
descriptor_2                          :: A third level of detail about the Problem (formerly Complaint Type) beyond the Problem Detail (formerly Descriptor). This is not used by every categor
status                                :: Status of SR submitted 



In [31]:
def by_type(where):
    response = httpx.get(url, params={
        "$select": "complaint_type, count(*) as n",
        "$where": where,
        "$group": "complaint_type",
        "$order": "n DESC",
        "$limit": "500",
    }, timeout=60.0)
    response.raise_for_status()
    return {row["complaint_type"]: int(row["n"]) for row in response.json()}

event_types, quiet_types = by_type(event), by_type(quiet)

types = sorted(set(event_types) | set(quiet_types))
compare = pd.DataFrame(
    [(t, event_types.get(t, 0) / 8, quiet_types.get(t, 0) / 7) for t in types],
    columns=["complaint_type", "event_day", "quiet_day"],
)
compare["ratio"] = (compare["event_day"]
                    / compare["quiet_day"].where(compare["quiet_day"] > 0)).round(2)
print(compare.sort_values("event_day", ascending=False).head(20).to_string(index=False))
print("\nonly in event week:", sorted(set(event_types) - set(quiet_types))[:10])
print("only in quiet week:", sorted(set(quiet_types) - set(event_types))[:10])

          complaint_type  event_day   quiet_day  ratio
          HEAT/HOT WATER   2851.375 1045.000000   2.73
         Illegal Parking   1088.125 1443.285714   0.75
     Noise - Residential    653.000  778.142857   0.84
        Blocked Driveway    466.750  478.000000   0.98
             Snow or Ice    391.125    0.428571 912.62
    UNSANITARY CONDITION    383.875  316.714286   1.21
                PLUMBING    245.875  228.714286   1.08
           PAINT/PLASTER    229.125  197.142857   1.16
             DOOR/WINDOW    199.250  148.285714   1.34
              WATER LEAK    182.250  147.000000   1.24
       Abandoned Vehicle    153.625  201.428571   0.76
Traffic Signal Condition    134.625  284.857143   0.47
        Street Condition    132.875  211.714286   0.63
            Water System    127.500  158.428571   0.80
                 GENERAL    123.625   93.428571   1.32
                   Noise    115.875  163.142857   0.71
      Noise - Commercial    115.125  161.000000   0.72
          

In [32]:
def closed_rows(where):
    rows, offset = [], 0
    while True:
        response = httpx.get(url, params={
            "$select": "unique_key, created_date, closed_date, status, descriptor_2",
            "$where": where,
            "$order": "unique_key",
            "$limit": "20000",
            "$offset": str(offset),
        }, timeout=120.0)
        response.raise_for_status()
        batch = response.json()
        rows.extend(batch)
        if len(batch) < 20000:
            return rows
        offset += 20000

WINDOWS = {"event_2024-01-15_22": event, "quiet_2024-01-24_30": quiet}
pulls = {}
for label, window in WINDOWS.items():
    rows = closed_rows(heat + window + " and closed_date is not null")
    pulls[label] = rows
    payload = json.dumps(rows, indent=2)
    out_file = raw_dir / f"heat_closed_{label}.json"
    out_file.write_text(payload, encoding="utf-8")
    (raw_dir / f"heat_closed_{label}.meta.json").write_text(json.dumps({
        "url": url, "where": heat + window + " and closed_date is not null",
        "rows": len(rows), "sha256": hashlib.sha256(payload.encode("utf-8")).hexdigest(),
        "fetched_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }, indent=2), encoding="utf-8")
    part = pd.DataFrame(rows)
    part["resolution_hours"] = ((pd.to_datetime(part["closed_date"])
                                - pd.to_datetime(part["created_date"])).dt.total_seconds() / 3600)
    print(f"\n{label}   closed {len(part):>6,}   median {part['resolution_hours'].median():6.1f} h"
          f"   p90 {part['resolution_hours'].quantile(.9):6.1f} h")
    for cls in part["descriptor_2"].value_counts().index:
        sub = part.loc[part["descriptor_2"] == cls, "resolution_hours"]
        print(f"    {cls:<28} n={len(sub):>6,}  median {sub.median():6.1f} h  p90 {sub.quantile(.9):6.1f} h")


event_2024-01-15_22   closed 22,770   median   46.7 h   p90   80.4 h
    NO HEAT                      n=15,307  median   47.7 h  p90   80.9 h
    NO HEAT AND NO HOT WATER     n= 6,218  median   44.5 h  p90   78.7 h
    NO HOT WATER                 n= 1,245  median   42.9 h  p90   79.9 h

quiet_2024-01-24_30   closed  7,309   median   28.9 h   p90   62.3 h
    NO HEAT                      n= 3,915  median   29.5 h  p90   63.0 h
    NO HEAT AND NO HOT WATER     n= 2,491  median   27.7 h  p90   60.2 h
    NO HOT WATER                 n=   903  median   29.0 h  p90   64.3 h


In [33]:
def weighted_median(values, weights):
    order = np.argsort(values)
    values, weights = np.asarray(values)[order], np.asarray(weights)[order]
    return float(values[np.searchsorted(np.cumsum(weights), weights.sum() / 2)])

event_part = pd.DataFrame(pulls["event_2024-01-15_22"])
quiet_part = pd.DataFrame(pulls["quiet_2024-01-24_30"])
for part in (event_part, quiet_part):
    part["resolution_hours"] = (pd.to_datetime(part["closed_date"]) - pd.to_datetime(part["created_date"])).dt.total_seconds() / 3600
event_shares = event_part["descriptor_2"].value_counts(normalize=True)
quiet_shares = quiet_part["descriptor_2"].value_counts(normalize=True)
quiet_part["event_weight"] = (quiet_part["descriptor_2"].map(event_shares)
                             / quiet_part["descriptor_2"].map(quiet_shares))
achieved = (quiet_part.groupby("descriptor_2")["event_weight"].sum()
            / quiet_part["event_weight"].sum())
print(pd.DataFrame({"target_event_mix": event_shares, "achieved": achieved}).round(3).to_string())
reweighted = weighted_median(quiet_part["resolution_hours"], quiet_part["event_weight"])
quiet_median = quiet_part["resolution_hours"].median()
event_median = event_part["resolution_hours"].median()
gap = event_median - quiet_median
mix_hours = reweighted - quiet_median
print(f"event median:           {event_median:.1f} h")
print(f"quiet median:           {quiet_median:.1f} h")
print(f"gap vs quiet pooled:     {gap:.1f} h")
print(f"explained by mix alone:  {mix_hours:.1f} h ({mix_hours / gap:.0%} of the gap)")
print("Interpretation: symptom mix explains only part of the descriptive gap; this does not establish causation.")

                          target_event_mix  achieved
descriptor_2                                        
NO HEAT                              0.672     0.672
NO HEAT AND NO HOT WATER             0.273     0.273
NO HOT WATER                         0.055     0.055
event median:           46.7 h
quiet median:           28.9 h
gap vs quiet pooled:     17.8 h
explained by mix alone:  0.1 h (1% of the gap)
Interpretation: symptom mix explains only part of the descriptive gap; this does not establish causation.


In [34]:
for label in ("event_2024-01-15_22", "quiet_2024-01-24_30"):
    p = raw_dir / f"heat_closed_{label}.meta.json"
    m = json.loads(p.read_text(encoding="utf-8"))
    m.update({
        "select": "unique_key, created_date, closed_date, status, descriptor_2",
        "order": "unique_key (stable for paging)",
        "paging": "$limit 20000, $offset until a short page",
        "population": "ALL closed HEAT/HOT WATER tickets created in the window — not a sample",
        "limitation": ("closed tickets only; safe here because ~99.8% of window intake is closed. "
                       "Biased on recent months, where slow tickets are still open."),
    })
    p.write_text(json.dumps(m, indent=2), encoding="utf-8")
    print(label, len(m), "fields")

event_2024-01-15_22 10 fields
quiet_2024-01-24_30 10 fields
